In [1]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmp8st36eoy".


## Matrix Multiplication using WMMA API

In [ ]:
%%cuda

#include <stdio.h>
#include <cuda.h>
#include <cuda_fp16.h>
#include <mma.h>

#define N 2048
#define TILE_SIZE 16 // Using 16x16 matrix tiles

using namespace nvcuda;

__global__ void matMulTensorCores(half *A, half *B, half *C) {
    int warpM = (blockIdx.x * blockDim.x + threadIdx.x) / warpSize;
    int warpN = (blockIdx.y * blockDim.y + threadIdx.y);

    if (warpM < N / TILE_SIZE && warpN < N / TILE_SIZE) {
        wmma::fragment<wmma::matrix_a, TILE_SIZE, TILE_SIZE, TILE_SIZE, half, wmma::row_major> fragA;
        wmma::fragment<wmma::matrix_b, TILE_SIZE, TILE_SIZE, TILE_SIZE, half, wmma::col_major> fragB;
        wmma::fragment<wmma::accumulator, TILE_SIZE, TILE_SIZE, TILE_SIZE, half> fragC;

        wmma::fill_fragment(fragC, __float2half(0.0f));

        for (int k = 0; k < N / TILE_SIZE; k++) {
            wmma::load_matrix_sync(fragA, A + warpM * TILE_SIZE * N + k * TILE_SIZE, N);
            wmma::load_matrix_sync(fragB, B + k * TILE_SIZE * N + warpN * TILE_SIZE, N);
            wmma::mma_sync(fragC, fragA, fragB, fragC);
        }

        wmma::store_matrix_sync(C + warpM * TILE_SIZE * N + warpN * TILE_SIZE, fragC, N, wmma::mem_row_major);
    }
}

void randomMatrix(half *mat, int size) {
    for (int i = 0; i < size; i++) {
        mat[i] = __float2half(static_cast<float>(rand()) / RAND_MAX);
    }
}

int main() {
    size_t bytes = N * N * sizeof(half);

    half *h_A, *h_B, *h_C;
    half *d_A, *d_B, *d_C;

    h_A = (half*)malloc(bytes);
    h_B = (half*)malloc(bytes);
    h_C = (half*)malloc(bytes);

    randomMatrix(h_A, N * N);
    randomMatrix(h_B, N * N);

    cudaMalloc(&d_A, bytes);
    cudaMalloc(&d_B, bytes);
    cudaMalloc(&d_C, bytes);

    cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice);

    dim3 blockSize(32, 32);
    dim3 gridSize(N / TILE_SIZE, N / TILE_SIZE);

    matMulTensorCores<<<gridSize, blockSize>>>(d_A, d_B, d_C);

    cudaMemcpy(h_C, d_C, bytes, cudaMemcpyDeviceToHost);

    printf("Matrix multiplication completed using Tensor Cores (FP16).\n");

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    free(h_A);
    free(h_B);
    free(h_C);

    return 0;
}


/tmp/tmp8st36eoy/735fe80d-ae8d-4027-be0f-66dffa05ec2d/single_file.cu(7): error: name must be a namespace name
  using namespace nvcuda;
                  ^

/tmp/tmp8st36eoy/735fe80d-ae8d-4027-be0f-66dffa05ec2d/single_file.cu(11): error: name followed by "::" must be a class or namespace name
      wmma::fragment<wmma::matrix_a, 16, 16, 16, half, wmma::col_major> a_frag;
      ^

/tmp/tmp8st36eoy/735fe80d-ae8d-4027-be0f-66dffa05ec2d/single_file.cu(11): error: type name is not allowed
      wmma::fragment<wmma::matrix_a, 16, 16, 16, half, wmma::col_major> a_frag;
                                                 ^

/tmp/tmp8st36eoy/735fe80d-ae8d-4027-be0f-66dffa05ec2d/single_file.cu(11): error: name followed by "::" must be a class or namespace name
      wmma::fragment<wmma::matrix_a, 16, 16, 16, half, wmma::col_major> a_frag;
                                                       ^

/tmp/tmp8st36eoy/735fe80d-ae8d-4027-be0f-66dffa05ec2d/single_file.cu(11): error: identifier "a_frag" is 

## Matrix Multiplication using tensor cores part 2

In [ ]:
%%cuda
#include <cstdio>
#include <cstdlib>
#include <cuda_fp16.h>
#include <mma.h>
#include <iostream>

using namespace nvcuda::wmma;

// Define matrix dimension N (must be multiple of 16)
#ifndef N
#define N 1024
#endif

// We will use 16×16 WMMA tiles
#define TILE_SIZE 16

// Kernel: half-precision inputs, float accumulation/output
//  1D block of 128 threads => 4 warps per block
//  Each warp computes one 16×16 tile in the output
__global__ void matMulTensorCoresFp32(const half* A, const half* B, float* C, int n)
{
    // warpsPerBlock = 4 if blockDim.x=128
    int warpsPerBlock = blockDim.x / 32;
    // global warp ID
    int warpId = blockIdx.x * warpsPerBlock + (threadIdx.x / 32);

    int tilesPerDim = n / TILE_SIZE;               // e.g. N=256 => 16
    int tileCount   = tilesPerDim * tilesPerDim;   // e.g. 16*16=256

    if (warpId >= tileCount) return;

    // Identify which 16×16 tile we're responsible for
    int tileRow = warpId / tilesPerDim;
    int tileCol = warpId % tilesPerDim;

    // Create WMMA fragments
    // matrix_a uses half in col_major
    fragment<matrix_a, TILE_SIZE, TILE_SIZE, TILE_SIZE, half, col_major> aFrag;
    // matrix_b uses half in row_major
    fragment<matrix_b, TILE_SIZE, TILE_SIZE, TILE_SIZE, half, row_major> bFrag;
    // accumulator uses float
    fragment<accumulator, TILE_SIZE, TILE_SIZE, TILE_SIZE, float> cFrag;

    // Initialize accumulator to 0.f
    fill_fragment(cFrag, 0.0f);

    // Loop over the K dimension in steps of 16
    for (int k0 = 0; k0 < n; k0 += TILE_SIZE)
    {
        const half* Ablock = A + (tileRow * TILE_SIZE * n) + k0;
        const half* Bblock = B + (k0 * n) + (tileCol * TILE_SIZE);

        load_matrix_sync(aFrag, Ablock, n);
        load_matrix_sync(bFrag, Bblock, n);

        // Multiply-accumulate
        mma_sync(cFrag, aFrag, bFrag, cFrag);
    }

    // Write the 16×16 tile of results to the float matrix C
    float* Csub = C + (tileRow * TILE_SIZE * n) + (tileCol * TILE_SIZE);
    store_matrix_sync(Csub, cFrag, n, mem_row_major);
}


// Simple helper: fill a half-precision matrix with random values [0..1]
void randomMatrix(half *mat, int size) {
    for (int i = 0; i < size; i++) {
        float r = static_cast<float>(rand()) / RAND_MAX;
        mat[i] = __float2half(r);
    }
}

// CPU reference multiply, but store result in float
//   C[i,j] = sum_{k} ( float(A[i,k]) * float(B[k,j]) )
void cpuMultiplyHalfFloat(const half* A, const half* B, float* C, int n) {
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            float sum = 0.0f;
            for (int k = 0; k < n; k++) {
                float aVal = __half2float(A[i * n + k]);
                float bVal = __half2float(B[k * n + j]);
                sum += aVal * bVal;
            }
            C[i * n + j] = sum;
        }
    }
}



int main()
{
    // N must be multiple of 16
    if (N % TILE_SIZE != 0) {
        std::cerr << "Error: N must be a multiple of 16.\n";
        return 1;
    }

    srand(0);

    // Host memory: half inputs, float output
    size_t totalElements = N * N;
    size_t bytesAorB     = totalElements * sizeof(half);
    size_t bytesC        = totalElements * sizeof(float);

    half*  h_A = (half*) malloc(bytesAorB);
    half*  h_B = (half*) malloc(bytesAorB);
    float* h_C = (float*) malloc(bytesC);    // GPU result
    float* h_Ccpu = (float*) malloc(bytesC); // CPU reference

    // Random initialization of A, B in half
    randomMatrix(h_A, N * N);
    randomMatrix(h_B, N * N);

    // Allocate device memory
    half*  d_A = nullptr;
    half*  d_B = nullptr;
    float* d_C = nullptr;
    cudaMalloc(&d_A, bytesAorB);
    cudaMalloc(&d_B, bytesAorB);
    cudaMalloc(&d_C, bytesC);

    // Copy A, B to device
    cudaMemcpy(d_A, h_A, bytesAorB, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, bytesAorB, cudaMemcpyHostToDevice);
    cudaMemset(d_C, 0, bytesC);

    // Compute grid/block configuration
    //  - 128 threads => 4 warps per block
    //  - total 16×16 tiles = (N/16)*(N/16)
    //  - each block handles 4 tiles
    int tilesPerDim  = N / TILE_SIZE;
    int tileCount    = tilesPerDim * tilesPerDim;
    int warpsPerBlk  = 128 / 32;  // 4
    int blocksNeeded = (tileCount + warpsPerBlk - 1) / warpsPerBlk;

    dim3 blockSize(128);
    dim3 gridSize(blocksNeeded);

    // Launch kernel
    matMulTensorCoresFp32<<<gridSize, blockSize>>>(d_A, d_B, d_C, N);
    cudaDeviceSynchronize();

    // Copy result back to host
    cudaMemcpy(h_C, d_C, bytesC, cudaMemcpyDeviceToHost);

    // CPU reference multiply (in float)
    cpuMultiplyHalfFloat(h_A, h_B, h_Ccpu, N);

    // Compare first 10 elements
    std::cout << "Matrix multiplication (FP16 inputs -> FP32 output) completed.\n";
    std::cout << "Compare first 10 elements (CPU vs. GPU):\n";
    for (int i = 0; i < 10; i++) {
        float cpuVal = h_Ccpu[i];
        float gpuVal = h_C[i];
        std::cout << "C[" << i << "]: CPU=" << cpuVal
                  << ", GPU=" << gpuVal << std::endl;
    }

    // Cleanup
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    free(h_A);
    free(h_B);
    free(h_C);
    free(h_Ccpu);

    return 0;
}